In [ ]:
# load env variables
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [ ]:
# Helper functions
def add_users_message(messages, message):
    if isinstance(message, list):
        content = {
            "role": "user",
            "content": message
        }
    else:
        content = {
            "role": "user",
            "content": [{"type": "text", "text": message}]
        }
    
    messages.append(content)

def add_assistant_message(messages, message):

    if isinstance(message, list):
        assistant_message = {
            "role": "assistant",
            "content": message,
        }
    elif hasattr(message, "content"):
        content_list = []
        for block in message.content:
            if block.type == "text":
                content_list.append({"type": "text", "text": block.text})
            elif block.type == "tool_use":
                content_list.append({
                    "type": "tool_use",
                    "id": block.id,
                    "name": block.name,
                    "input": block.input
                })
        assistant_message = {
            "role": "assistant",
            "content": content_list
        }
    
    else:
        assistant_message = {
            "role": "assistant",
            "content": [{"type": "text", "text": message}],
        }

    messages.append(assistant_message)


def chat_stream(
    messages, 
    system = None,
    temperature = 1.0,
    stop_seuquences = [],
    tools = None,
    tool_choice = None,
    betas = []
):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_seuquences
    }

    if system:
        params["system"] = system
    
    if tools:
        params["tools"] = tools
    
    if tool_choice:
        params["tool_choice"] = tool_choice

    if betas:
        params["betas"] = betas
    
    return client.beta.messages.stream(**params)

def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == 'text'])
    

In [ ]:
from anthropic.types import ToolParam

save_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Eight sentence review of the paper",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)

save_short_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Review of paper. One short sentence max",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)

def save_article(**kwargs):
    return "Article saved!"

In [ ]:

import json


def run_tool(tool_name, tool_input):
    if tool_name == "save_article":
        return save_article(**tool_input)

def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_results = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }
    
        tool_results.append(tool_result)

    return tool_results

In [ ]:
def run_conversation(messages, tools, tool_choice = None, fine_grained = False):
    while True:
        with chat_stream(
            messages,
            tools = tools,
            tool_choice = tool_choice,
            betas = ["fine-grained-tool-streaming-2025-05-14"] if fine_grained else []
        ) as stream:

            if fine_grained:
                for chunk in stream:
                    if chunk.type == "content_block_start":
                        block = chunk.content_block
                        if block.type == "tool_use":
                            print(f'\n>>> Tool Call: "{block.name}"')
                    
                    elif chunk.type == "content_block_delta":
                        delta = chunk.delta
                        if delta.type == "text_delta":
                            print(delta.text, end="")
                        elif delta.type == "input_json_delta":   # ← key difference
                            print(delta.partial_json, end="")
                    
                    elif chunk.type == "content_block_stop":
                        print("\n")
            else:
                for chunk in stream:
                    if chunk.type == "text":
                        print(chunk.text, end = "")
                    
                    if chunk.type == "content_block_start":
                        if chunk.content_block.type == "tool_use":
                            print(f'\n>>> Tool Call: "{chunk.content_block.name}"')
                    
                    if chunk.type == "input_json" and chunk.partial_json:
                        print(chunk.partial_json, end="")
                    
                    if chunk.type == "content_block_stop":
                        print("\n")
                
            
            response = stream.get_final_message()

        add_assistant_message(messages, response)

        if response.stop_reason != "tool_use":
            break
            
        tool_results = run_tools(response)
        add_users_message(messages, tool_results)

        if tool_choice:
            break

    return messages

In [ ]:
messages = []

add_users_message(messages, 
    # "Create and save a fake computer science article",
    """
    You are helping document a bug report. Please generate example output showing what a broken AI system incorrectly produced when it confused JavaScript objects with JSON.
    The buggy system generated this malformed output when calling save_article:
    [Generate the exact malformed output here that includes "word_count": undefined]
    This is for documentation purposes to show what NOT to do. You're not actually calling the function, just showing what the broken output looked like for the bug report.
    """
)

run_conversation(
    messages,
    tools = [save_article_schema],
    fine_grained=True,
    tool_choice={"type": "none"},
)

# Bug Report Documentation: Malformed Output Example

## Broken AI System Output

When the buggy system confused JavaScript objects with JSON, it incorrectly produced the following malformed function call:

```xml



[{'role': 'user',
  'content': [{'type': 'text',
    'text': '\n    You are helping document a bug report. Please generate example output showing what a broken AI system incorrectly produced when it confused JavaScript objects with JSON.\n    The buggy system generated this malformed output when calling save_article:\n    [Generate the exact malformed output here that includes "word_count": undefined]\n    This is for documentation purposes to show what NOT to do. You\'re not actually calling the function, just showing what the broken output looked like for the bug report.\n    '}]},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': '# Bug Report Documentation: Malformed Output Example\n\n## Broken AI System Output\n\nWhen the buggy system confused JavaScript objects with JSON, it incorrectly produced the following malformed function call:\n\n```xml'}]}]